# Python + RDBMS + SQL Training  
## Notebook 3 of 3: Python Database Integration and Final Project

### Duration
This notebook is designed for approximately **4 hours** of hands-on learning.

### What you will learn
- Connect Python with SQLite
- Create database tables from Python
- Build reusable Python database functions
- Perform CRUD operations
- Use parameterized SQL
- Use Pandas for SQL reports
- Export reports to CSV
- Use transactions
- Build an end-to-end mini project


## 1. Project Overview

We will build a **Student Course Enrollment Management System**.

### Features
- Add student
- Add instructor
- Add course
- Enroll student
- Record payment
- Search student
- Update records
- Cancel enrollment
- Generate reports
- Export reports to CSV

### Tables
1. students
2. instructors
3. courses
4. enrollments
5. payments
6. attendance


## 2. Python + Database Workflow

```text
1. Connect to database
2. Create tables
3. Insert data
4. Define helper functions
5. Perform CRUD
6. Write analytical SQL
7. Convert query output to DataFrame
8. Export reports
```


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DB_NAME = "student_course_project.db"
path = Path(DB_NAME)
if path.exists():
    path.unlink()

conn = sqlite3.connect(DB_NAME)
conn.execute("PRAGMA foreign_keys = ON;")
print("Database created:", DB_NAME)


## 3. Helper Functions


In [ ]:
def exec_script(script):
    conn.executescript(script)
    conn.commit()
    print("Script executed successfully.")

def execute(sql, params=None):
    if params is None:
        params = []
    cur = conn.execute(sql, params)
    conn.commit()
    print(f"Rows affected: {cur.rowcount}")
    return cur

def query(sql, params=None):
    if params is None:
        params = []
    return pd.read_sql_query(sql, conn, params=params)

def show_table(table_name):
    return query(f"SELECT * FROM {table_name};")

def show_schema(table_name):
    print(f"\nSchema: {table_name}")
    display(query(f"PRAGMA table_info({table_name});"))
    fk = query(f"PRAGMA foreign_key_list({table_name});")
    if not fk.empty:
        print("Foreign keys:")
        display(fk)


## 4. Create Project Tables

This schema uses:
- Primary keys
- Foreign keys
- Unique constraints
- Check constraints
- Auto-increment IDs


In [ ]:
exec_script("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    city TEXT,
    registration_date DATE NOT NULL
);

CREATE TABLE instructors (
    instructor_id INTEGER PRIMARY KEY AUTOINCREMENT,
    instructor_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    specialization TEXT NOT NULL
);

CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    course_name TEXT NOT NULL,
    instructor_id INTEGER NOT NULL,
    fee REAL NOT NULL CHECK (fee >= 0),
    duration_hours INTEGER NOT NULL CHECK (duration_hours > 0),
    level TEXT NOT NULL CHECK (level IN ('Beginner', 'Intermediate', 'Advanced')),
    FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id)
);

CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    enrollment_date DATE NOT NULL,
    status TEXT NOT NULL CHECK (status IN ('Active', 'Completed', 'Cancelled')),
    UNIQUE(student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);

CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    enrollment_id INTEGER NOT NULL UNIQUE,
    amount REAL NOT NULL CHECK (amount >= 0),
    payment_date DATE NOT NULL,
    payment_status TEXT NOT NULL CHECK (payment_status IN ('Paid', 'Pending', 'Refunded')),
    FOREIGN KEY (enrollment_id) REFERENCES enrollments(enrollment_id)
);
""")


## 5. Inspect Schema


In [ ]:
for table_name in ["students", "instructors", "courses", "enrollments", "payments"]:
    show_schema(table_name)


## 6. Insert Initial Data


In [ ]:
exec_script("""
INSERT INTO instructors (instructor_name, email, specialization) VALUES
('Dr. Meera Iyer', 'meera.iyer@example.com', 'Data Science'),
('Arjun Sen', 'arjun.sen@example.com', 'Python Development'),
('Kavita Rao', 'kavita.rao@example.com', 'Business Analytics'),
('Ravi Menon', 'ravi.menon@example.com', 'Cloud Databases');

INSERT INTO students (student_name, email, city, registration_date) VALUES
('Rahul Kumar', 'rahul.kumar@example.com', 'Patna', '2026-01-05'),
('Priya Singh', 'priya.singh@example.com', 'Kolkata', '2026-01-06'),
('Amit Raj', 'amit.raj@example.com', 'Delhi', '2026-01-07'),
('Sneha Verma', 'sneha.verma@example.com', 'Patna', '2026-01-10'),
('Aditya Sharma', 'aditya.sharma@example.com', 'Mumbai', '2026-01-12'),
('Neha Gupta', 'neha.gupta@example.com', 'Ranchi', '2026-01-13');

INSERT INTO courses (course_name, instructor_id, fee, duration_hours, level) VALUES
('Python for Beginners', 2, 4999, 20, 'Beginner'),
('SQL and RDBMS Masterclass', 1, 6999, 24, 'Beginner'),
('Machine Learning Basics', 1, 11999, 36, 'Intermediate'),
('Business Dashboarding', 3, 8999, 30, 'Intermediate'),
('Cloud Database Fundamentals', 4, 10999, 28, 'Beginner');

INSERT INTO enrollments (student_id, course_id, enrollment_date, status) VALUES
(1, 1, '2026-02-01', 'Active'),
(1, 2, '2026-02-03', 'Completed'),
(2, 2, '2026-02-04', 'Active'),
(3, 3, '2026-02-05', 'Active'),
(4, 4, '2026-02-07', 'Cancelled'),
(5, 5, '2026-02-08', 'Active'),
(6, 1, '2026-02-08', 'Completed');

INSERT INTO payments (enrollment_id, amount, payment_date, payment_status) VALUES
(1, 4999, '2026-02-01', 'Paid'),
(2, 6999, '2026-02-03', 'Paid'),
(3, 6999, '2026-02-04', 'Pending'),
(4, 11999, '2026-02-05', 'Paid'),
(5, 0, '2026-02-07', 'Refunded'),
(6, 10999, '2026-02-08', 'Paid'),
(7, 4999, '2026-02-08', 'Paid');
""")


## 7. View Initial Tables


In [ ]:
show_table("students")


In [ ]:
show_table("courses")


In [ ]:
show_table("enrollments")


## 8. CRUD Function: Add Student


In [ ]:
def add_student(student_name, email, city, registration_date):
    execute("""
    INSERT INTO students (student_name, email, city, registration_date)
    VALUES (?, ?, ?, ?);
    """, (student_name, email, city, registration_date))

add_student("Rohan Das", "rohan.das@example.com", "Pune", "2026-02-15")
query("SELECT * FROM students WHERE email = ?;", ("rohan.das@example.com",))


## 9. CRUD Function: Add Instructor


In [ ]:
def add_instructor(instructor_name, email, specialization):
    execute("""
    INSERT INTO instructors (instructor_name, email, specialization)
    VALUES (?, ?, ?);
    """, (instructor_name, email, specialization))

add_instructor("Mansi Roy", "mansi.roy@example.com", "MLOps")
show_table("instructors")


## 10. CRUD Function: Add Course


In [ ]:
def add_course(course_name, instructor_id, fee, duration_hours, level):
    execute("""
    INSERT INTO courses (course_name, instructor_id, fee, duration_hours, level)
    VALUES (?, ?, ?, ?, ?);
    """, (course_name, instructor_id, fee, duration_hours, level))

add_course("MLOps Fundamentals", 5, 13999, 32, "Advanced")
show_table("courses")


## 11. CRUD Function: Enroll Student


In [ ]:
def enroll_student(student_id, course_id, enrollment_date, status="Active"):
    execute("""
    INSERT INTO enrollments (student_id, course_id, enrollment_date, status)
    VALUES (?, ?, ?, ?);
    """, (student_id, course_id, enrollment_date, status))

enroll_student(7, 6, "2026-02-16", "Active")
show_table("enrollments")


## 12. CRUD Function: Record Payment


In [ ]:
def record_payment(enrollment_id, amount, payment_date, payment_status):
    execute("""
    INSERT INTO payments (enrollment_id, amount, payment_date, payment_status)
    VALUES (?, ?, ?, ?);
    """, (enrollment_id, amount, payment_date, payment_status))

record_payment(8, 13999, "2026-02-16", "Pending")
show_table("payments")


## 13. Search Student by Name


In [ ]:
def search_students(keyword):
    return query("""
    SELECT *
    FROM students
    WHERE LOWER(student_name) LIKE LOWER(?)
    ORDER BY student_name;
    """, (f"%{keyword}%",))

search_students("ra")


## 14. Update Student City


In [ ]:
def update_student_city(student_id, new_city):
    execute("""
    UPDATE students
    SET city = ?
    WHERE student_id = ?;
    """, (new_city, student_id))

update_student_city(7, "Bengaluru")
query("SELECT * FROM students WHERE student_id = 7;")


## 15. Cancel Enrollment


In [ ]:
def cancel_enrollment(enrollment_id):
    execute("""
    UPDATE enrollments
    SET status = 'Cancelled'
    WHERE enrollment_id = ?;
    """, (enrollment_id,))

cancel_enrollment(8)
query("SELECT * FROM enrollments WHERE enrollment_id = 8;")


## 16. Duplicate Enrollment Protection


In [ ]:
try:
    enroll_student(1, 1, "2026-03-01", "Active")
except Exception as e:
    print("Duplicate enrollment blocked:", e)


## 17. Create Reporting View


In [ ]:
exec_script("""
DROP VIEW IF EXISTS vw_enrollment_report;

CREATE VIEW vw_enrollment_report AS
SELECT
    e.enrollment_id,
    s.student_id,
    s.student_name,
    s.email AS student_email,
    s.city,
    c.course_id,
    c.course_name,
    c.level,
    c.fee,
    c.duration_hours,
    i.instructor_name,
    e.enrollment_date,
    e.status AS enrollment_status,
    p.amount,
    p.payment_date,
    p.payment_status
FROM enrollments e
INNER JOIN students s
    ON e.student_id = s.student_id
INNER JOIN courses c
    ON e.course_id = c.course_id
INNER JOIN instructors i
    ON c.instructor_id = i.instructor_id
LEFT JOIN payments p
    ON e.enrollment_id = p.enrollment_id;
""")

query("SELECT * FROM vw_enrollment_report ORDER BY enrollment_id;")


## 18. Report: Student Course Mapping


In [ ]:
student_course_report = query("""
SELECT
    student_name,
    city,
    course_name,
    instructor_name,
    enrollment_date,
    enrollment_status
FROM vw_enrollment_report
ORDER BY student_name, course_name;
""")
student_course_report


## 19. Report: Revenue by Course


In [ ]:
revenue_by_course = query("""
SELECT
    course_name,
    COUNT(enrollment_id) AS total_enrollments,
    SUM(CASE WHEN payment_status = 'Paid' THEN amount ELSE 0 END) AS paid_revenue,
    SUM(CASE WHEN payment_status = 'Pending' THEN amount ELSE 0 END) AS pending_amount,
    SUM(CASE WHEN payment_status = 'Refunded' THEN amount ELSE 0 END) AS refunded_amount
FROM vw_enrollment_report
GROUP BY course_name
ORDER BY paid_revenue DESC;
""")
revenue_by_course


## 20. Report: Pending Payments


In [ ]:
pending_payment_report = query("""
SELECT
    student_name,
    student_email,
    course_name,
    amount,
    payment_date,
    payment_status
FROM vw_enrollment_report
WHERE payment_status = 'Pending'
ORDER BY amount DESC;
""")
pending_payment_report


## 21. Report: Instructor Performance


In [ ]:
instructor_performance = query("""
SELECT
    instructor_name,
    COUNT(enrollment_id) AS total_enrollments,
    SUM(CASE WHEN payment_status = 'Paid' THEN amount ELSE 0 END) AS paid_revenue
FROM vw_enrollment_report
GROUP BY instructor_name
ORDER BY paid_revenue DESC;
""")
instructor_performance


## 22. Export Reports to CSV


In [ ]:
student_course_report.to_csv("student_course_report.csv", index=False)
revenue_by_course.to_csv("revenue_by_course.csv", index=False)
pending_payment_report.to_csv("pending_payment_report.csv", index=False)
instructor_performance.to_csv("instructor_performance.csv", index=False)

print("CSV reports exported successfully.")


## 23. Visualize Revenue by Course


In [ ]:
chart_df = revenue_by_course.sort_values("paid_revenue", ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(chart_df["course_name"], chart_df["paid_revenue"])
plt.xlabel("Paid Revenue")
plt.ylabel("Course")
plt.title("Paid Revenue by Course")
plt.show()


## 24. Transaction-Based Enrollment with Payment

Enrollment and payment should be saved together.

If payment fails, enrollment should also roll back.


In [ ]:
def enroll_with_payment(student_id, course_id, enrollment_date, status, amount, payment_date, payment_status):
    try:
        conn.execute("BEGIN;")

        cur = conn.execute("""
        INSERT INTO enrollments (student_id, course_id, enrollment_date, status)
        VALUES (?, ?, ?, ?);
        """, (student_id, course_id, enrollment_date, status))

        enrollment_id = cur.lastrowid

        conn.execute("""
        INSERT INTO payments (enrollment_id, amount, payment_date, payment_status)
        VALUES (?, ?, ?, ?);
        """, (enrollment_id, amount, payment_date, payment_status))

        conn.commit()
        print("Transaction successful. Enrollment ID:", enrollment_id)
    except Exception as e:
        conn.rollback()
        print("Transaction failed and rolled back:", e)

enroll_with_payment(2, 6, "2026-03-05", "Active", 13999, "2026-03-05", "Paid")
query("SELECT * FROM vw_enrollment_report WHERE student_id = 2 ORDER BY enrollment_id;")


## 25. Add Attendance Table

This extends the project with another related table.


In [ ]:
exec_script("""
CREATE TABLE attendance (
    attendance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    enrollment_id INTEGER NOT NULL,
    session_date DATE NOT NULL,
    attendance_status TEXT NOT NULL CHECK (attendance_status IN ('Present', 'Absent')),
    FOREIGN KEY (enrollment_id) REFERENCES enrollments(enrollment_id)
);

INSERT INTO attendance (enrollment_id, session_date, attendance_status) VALUES
(1, '2026-03-01', 'Present'),
(1, '2026-03-02', 'Present'),
(1, '2026-03-03', 'Absent'),
(2, '2026-03-01', 'Present'),
(2, '2026-03-02', 'Present'),
(3, '2026-03-01', 'Absent'),
(3, '2026-03-02', 'Present'),
(4, '2026-03-01', 'Present'),
(4, '2026-03-02', 'Present'),
(4, '2026-03-03', 'Present');
""")
show_table("attendance")


## 26. Attendance Percentage Report


In [ ]:
attendance_report = query("""
SELECT
    r.student_name,
    r.course_name,
    COUNT(a.attendance_id) AS total_sessions,
    SUM(CASE WHEN a.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_sessions,
    ROUND(
        100.0 * SUM(CASE WHEN a.attendance_status = 'Present' THEN 1 ELSE 0 END) / COUNT(a.attendance_id),
        2
    ) AS attendance_percentage
FROM attendance a
INNER JOIN vw_enrollment_report r
    ON a.enrollment_id = r.enrollment_id
GROUP BY r.student_name, r.course_name
ORDER BY attendance_percentage DESC;
""")
attendance_report


## 27. Export Final Project Reports


In [ ]:
attendance_report.to_csv("attendance_report.csv", index=False)
show_table("students").to_csv("final_students.csv", index=False)
show_table("courses").to_csv("final_courses.csv", index=False)
show_table("enrollments").to_csv("final_enrollments.csv", index=False)
show_table("payments").to_csv("final_payments.csv", index=False)
show_table("attendance").to_csv("final_attendance.csv", index=False)

print("Final project files exported.")


## 28. Final Assignment

Build one of the following extensions.

### Option A: Course Feedback
Create table:
```sql
feedback (
    feedback_id,
    enrollment_id,
    rating,
    comments,
    feedback_date
)
```
Requirements:
- Rating must be between 1 and 5.
- Generate average rating per course.
- Find courses with rating below 3.

### Option B: Library Management System
Create:
- books
- members
- issued_books
- returns

Requirements:
- Use primary and foreign keys.
- Insert sample data.
- Write 15 queries.
- Export one CSV report.

### Option C: Sales Order Management System
Create:
- customers
- products
- orders
- order_items
- payments

Requirements:
- Use joins.
- Use aggregation.
- Use at least one CTE.
- Generate product-wise revenue report.


## 29. Final SQL Practice Questions

Write SQL queries for:

1. Show all students with their enrolled courses.
2. Show revenue by course.
3. Show revenue by instructor.
4. Show students with pending payments.
5. Show course-wise attendance percentage.
6. Show students with attendance below 75%.
7. Show city-wise student count.
8. Show course with highest enrollment.
9. Show top 3 students by paid amount.
10. Show cancelled enrollments.
11. Show active enrollments.
12. Show courses with no pending payment.
13. Show average fee by course level.
14. Show students enrolled in more than one course.
15. Show complete student-course-payment-attendance summary.


## 30. Final Assessment Checklist

Your final solution should contain:

- Minimum 4 related tables
- Primary keys
- Foreign keys
- Unique constraint
- Check constraint
- Sample data
- CRUD operations from Python
- 15 SQL queries
- Join queries
- Aggregation queries
- At least one subquery
- At least one CTE
- CSV export


## 31. Notebook 3 Summary

You completed:

- Python database connection
- Schema creation
- CRUD functions
- Parameterized queries
- Transaction handling
- SQL reporting
- Pandas integration
- CSV exports
- Attendance extension
- Final project assignment

This completes the 12-hour Python + RDBMS + SQL hands-on training.
